# How the selected model solves the home-price task

This is a teaching notebook, not a copy of the production source. It explains the complete path from monthly sales data to a predicted price. Each small snippet is immediately followed by what its output means and where the model uses it.

The two tasks are:

- **Off-market:** predict sale price without seeing list price.
- **List-aware:** predict sale price with list price available.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'gbt_ensemble_experiments').is_dir():
    ROOT = ROOT.parent
RESULTS = ROOT / 'gbt_ensemble_experiments' / 'results'


## 1. The whole pipeline first

One row represents one home. Raw columns such as area, bedrooms, community, coordinates, date, and remarks are converted into a row of model inputs. Earlier sold prices are also summarized into leakage-safe local-history columns. Three models then produce three candidate prices, which are blended into the final answer.

```text
earlier sold homes ──> physical/location/time features ──┐
                   └─> local price-history features ─────┼─> tree model ──> price A
remarks + structured features ──────────────────────────┼─> text model ──> price B
same engineered table, different boosted trees ─────────┼─> tree model ──> price C
                                                          └─> weighted blend ─> final price
```

The local-history encoder is therefore a **feature maker**, not the final estimator. A value such as `encoded` becomes one column in the table consumed by LightGBM or XGBoost.


### What is in each final ensemble?

| Task | Component | Training history | Special information | Blend weight |
|---|---|---|---|---:|
| Off-market | recency LightGBM | all earlier sales | regular local history + 24-month recency history | 15% |
| Off-market | word Ridge | latest 36 months | remarks TF-IDF + structured features | 20% |
| Off-market | XGBoost | all earlier sales | regular local history + 4-month recency history | 65% |
| List-aware | regular LightGBM | latest 36 months | list price + regular local history | 35% |
| List-aware | recency LightGBM | latest 36 months | list price + regular and 4-month recency history | 35% |
| List-aware | word Ridge | latest 36 months | list price + remarks TF-IDF + structured features | 30% |

`all earlier sales` means January 2021 through the month before the prediction cutoff. A recency **half-life** does not discard older rows: it reduces their influence inside the local-history averages.


## 2. How the months are fed to the model

Suppose we are validating March 2026. March homes are the homes to predict. Their sale prices are hidden from the feature builders. Only sales before March are permitted as training history.


In [6]:
dates = pd.date_range('2021-01-01', '2026-05-01', freq='MS') + pd.Timedelta(days=14)
listings = pd.DataFrame({'date_sold': dates})
cutoff = pd.Timestamp('2026-03-01')
march_homes = listings[listings.date_sold.dt.to_period('M') == pd.Period('2026-03')]
past_sales = listings[listings.date_sold < cutoff]
recent_sales = past_sales[past_sales.date_sold >= cutoff - pd.DateOffset(months=36)]


`past_sales` feeds the off-market tree components. `recent_sales` feeds all list-aware components and both text models. The target encoders are fitted from the same history as their associated tree; they never see March sale prices.

Walk-forward validation repeats this split six times: train before December and predict December, train before January and predict January, and so on through May. This tests whether the approach survives movement into the future.


In [7]:
validation_months = pd.period_range('2025-12', '2026-05', freq='M')
splits = []
for month in validation_months:
    train = listings[listings.date_sold < month.start_time]
    homes = listings[listings.date_sold.dt.to_period('M') == month]
    splits.append((str(month), len(train), len(homes)))
splits


[('2025-12', 59, 1),
 ('2026-01', 60, 1),
 ('2026-02', 61, 1),
 ('2026-03', 62, 1),
 ('2026-04', 63, 1),
 ('2026-05', 64, 1)]

For the real hidden June-September test, the final workflow is different: it fits each component **once** at the June 1 cutoff, saves the fitted objects, and reuses them for every test home. It does not retrain four times because the true June-August sale prices are unavailable. See [`train_final.py`](../train_final.py).


## 3. Turning one home into a feature row

The model cannot directly reason about a value such as `700-899` square feet. The feature builder converts it to useful numbers. Here the midpoint becomes `799.5`, then ratios give the model context about how that space is divided.


In [ ]:
low, high = map(float, '700-899'.split('-'))
area = (low + high) / 2
base_features = pd.DataFrame({
    'square_feet': [area],
    'area_per_bed': [area / 2],
    'bath_per_bed': [2 / 2],
})
base_features


This one-row table is the beginning of `X`, the model-input matrix. Production adds bedrooms, bathrooms, parking, garage, lot size, coordinates, time, property categories, and remarks flags. One row stays one home; each derived value becomes another column.

Location interactions let the model distinguish, for example, a condo in a community from a detached home in the same community.


In [ ]:
community = 'Willowdale East'
house_type = 'Condo Apt'
base_features['community_type'] = community + '|' + house_type
base_features['geo_cell'] = f'{43.77284:.2f}|{-79.39127:.2f}'
base_features


The model sees `community_type` and `geo_cell` as categories. It can learn different price patterns for different combinations without assuming that a category number has mathematical meaning. Time features similarly tell it both the long-term market period and the season.


In [ ]:
listed = pd.Timestamp('2025-11-15')
base_features['month_index'] = (listed.year - 2021) * 12 + listed.month - 1
base_features['month_sin'] = np.sin(2 * np.pi * listed.month / 12)
base_features['month_cos'] = np.cos(2 * np.pi * listed.month / 12)


The tree models receive several simple remarks flags too. This flag tells them that condition may be better than the physical columns alone suggest. The separate Ridge model later reads the full remarks text.


In [ ]:
remarks = 'Beautiful newly renovated kitchen near transit'.lower()
base_features['remark_renovated'] = int('renovat' in remarks)
base_features['remark_transit'] = int('transit' in remarks)


Production implementation: [`experiments/experiment.py`](../experiments/experiment.py), functions `square_feet` and `make_features`.


## 4. What `encoded` means and where it goes

Assume three earlier homes in the same community sold for $500k, $550k, and $600k. For the $550k training row, the encoder estimates the typical community log price while removing that row's own answer. `prior * global_mean` pulls small groups toward the overall market so one unusual sale does not dominate.


In [ ]:
local_log_prices = np.log(np.array([500_000, 550_000, 600_000]))
row_value = local_log_prices[1]
global_mean, prior = np.log(700_000), 5
encoded = (local_log_prices.sum() - row_value + prior * global_mean) / (len(local_log_prices) - 1 + prior)
encoded


The result is not used alone. It is attached to the same home's feature row as a new numeric column. Now the tree can combine facts such as “large condo” with “homes of this type in this community usually sell around this level.”


In [ ]:
local_features = pd.DataFrame({'te_log_price_community': [encoded]})
X_for_tree = pd.concat([base_features.reset_index(drop=True), local_features], axis=1)
X_for_tree[['square_feet', 'community_type', 'te_log_price_community']]


Production makes these summaries for six location groupings, including community, community plus property type, rounded coordinate cells, and exact shared coordinates. It summarizes log price and log price per square foot; list-aware models also get the usual log sold-to-list ratio.

For a future home there is no own answer to remove. The encoder looks up the full group summary made from allowed earlier sales. An unseen group falls back to the global value. See [`experiments/target_encoding_experiment.py`](../experiments/target_encoding_experiment.py), class `TargetEncoder`.


### Recency-weighted local history

A 4-month half-life means a sale from 4 months ago counts half as much as a current sale; after 8 months it counts one quarter. These weights are used only while calculating additional local-history columns.


In [ ]:
age_months = np.array([0, 4, 8, 12])
weights = 2 ** (-age_months / 4)
local_prices = np.array([700_000, 650_000, 600_000, 550_000])
recent_local_price = np.average(local_prices, weights=weights)
weights, recent_local_price


That weighted value becomes another column such as `recent4_log_price_community`. The regular encoder describes the stable local level; the recency encoder helps the model notice a changing market. See [`experiments/recency_encoding_experiment.py`](../experiments/recency_encoding_experiment.py), class `WeightedTargetEncoder`.


## 5. How the tree models use the constructed features

LightGBM and XGBoost build many small decision trees. A tree may first separate condos from detached homes, then split by floor area, local historical price, or market month. Each new tree learns a correction to the errors made by the trees already built. Their advantage here is learning non-linear combinations without us writing every rule.

This tiny ordinary tree only illustrates the interface. Production uses hundreds of boosted trees and the full feature table.


In [ ]:
from sklearn.tree import DecisionTreeRegressor
toy_X = pd.DataFrame({'area': [700, 900, 1500, 1800], 'local_log_price': np.log([500e3, 600e3, 900e3, 1.1e6])})
toy_y = np.log([510e3, 620e3, 920e3, 1.08e6])
tree = DecisionTreeRegressor(max_depth=2).fit(toy_X, toy_y)
predicted_price = np.exp(tree.predict(pd.DataFrame({'area': [1000], 'local_log_price': [np.log(650e3)]})))
predicted_price


For an off-market home, the tree output is predicted log sale price, which `exp` converts back to dollars. For a list-aware home, the output is the predicted log sale-to-list ratio, so the final component price is `list_price * exp(output)`.

The selected off-market LightGBM and XGBoost use similar inputs but different algorithms and recency settings. They make different errors, which is useful when blending. Production fitting is in [`train_final.py`](../train_final.py), functions `fit_lgb_component` and `fit_xgb_component`.


## 6. How the text Ridge model works

TF-IDF turns words and two-word phrases into numeric columns. A word receives a larger value when it is important in this remark but not common in every remark. The production text table joins up to 20,000 such columns with one-hot property categories and scaled numeric features. Ridge learns one linear weight per column.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
remarks = ['renovated kitchen near transit', 'as is fixer upper', 'luxury condo with lake view']
text_X = TfidfVectorizer(ngram_range=(1, 2)).fit_transform(remarks)
text_y = np.log([800_000, 550_000, 1_100_000])
text_model = Ridge(alpha=1.0).fit(text_X, text_y)


In the real task, positive and negative weights let the model adjust for language associated with condition, quality, and location while also considering structured features. Ridge is simpler than the tree models, but its different view of the data makes the ensemble more robust. See [`experiments/linear_text_experiment.py`](../experiments/linear_text_experiment.py), class `SparseFeatures`.


## 7. What each model is trained to predict

Prices are strongly skewed, so every component learns in log space. This makes an error from $500k to $550k more comparable to an error from $1.0M to $1.1M.


In [ ]:
sold_price, list_price = 900_000, 950_000
off_market_target = np.log(sold_price)
list_aware_target = np.log(sold_price / list_price)
off_market_price = np.exp(off_market_target)
list_aware_price = list_price * np.exp(list_aware_target)


The list-aware model therefore does not relearn the whole price. It starts from the known asking price and learns the likely percentage-like adjustment above or below it. Off-market models never receive list price.


## 8. From three component prices to one answer

Each component independently returns a price in dollars. They are combined with a weighted geometric mean, which is equivalent to averaging their log-price predictions.


In [ ]:
component_prices = np.array([820_000, 860_000, 840_000])
off_market_weights = np.array([0.15, 0.20, 0.65])
raw_blend = np.exp(np.sum(off_market_weights * np.log(component_prices)))
final_price = raw_blend * 0.9772745745864185
final_price


Here those prices mean recency LightGBM, text Ridge, and XGBoost. The final factor corrects a small systematic tendency to overpredict. List-aware uses its own three components, weights, and factor. Prediction from a saved checkpoint follows exactly this sequence in [`predict_artifact.py`](../predict_artifact.py).


## 9. How the models were tuned

Tuning was time-based, not a random train/test split. Candidate feature sets and model settings were trained on earlier data and compared mainly on November 2025 MdAPE. Experiments included history length, tree objective and complexity, target-encoding prior, recency half-life, number of boosting iterations, and Ridge regularization. Early stopping helped identify a useful tree count.

This is the basic experiment pattern; only the candidate setting changes.


In [ ]:
candidate_half_lives = [4, 8, 12, 24]
candidate_ridge_alpha = [1, 10, 50, 200]
# For each candidate: fit on sales before 2025-11-01, predict November,
# then prefer the lower median absolute percentage error (MdAPE).


This was a focused manual/grid search rather than an enormous automatic search. Feature ideas and alternative model families were promoted only when time-forward results improved. The final fixed settings include 628 trees for off-market recency LightGBM, 868 for off-market XGBoost, 127/198 for the two list-aware LightGBMs, and Ridge `alpha` values 1 and 50.

After selecting the three components, ensemble weights were searched separately using saved predictions for September-November 2025. Every weight combination on a 0.05 grid that summed to one was scored.


In [ ]:
weight_candidates = []
for first in range(21):
    for second in range(21 - first):
        weight_candidates.append((first / 20, second / 20, (20 - first - second) / 20))
len(weight_candidates), weight_candidates[:3]


For each candidate blend, calibration centers the typical log prediction error. If predictions are generally 2% too high, the learned factor is slightly below 1.


In [ ]:
actual = np.array([800_000, 900_000, 1_000_000])
raw_prediction = np.array([820_000, 930_000, 1_010_000])
calibration = np.exp(np.median(np.log(actual / raw_prediction)))
calibrated_prediction = raw_prediction * calibration
calibration


The weights and calibration factors were frozen from September-November, not optimized on the reported December-May scores. However, some finalist components were examined on December-May during development, so that period is time-forward validation but not a perfectly untouched test set. The hidden June-September competition set is the true unseen test.

The complete reproducible search is [`select_final_blend.py`](../select_final_blend.py). Its saved decision is loaded below.


In [ ]:
selection = json.loads((RESULTS / 'final_blend_selection.json').read_text())
pd.DataFrame({
    track: {'weights': values['weights'], 'calibration': values['calibration']}
    for track, values in selection.items()
}).T


## 10. One validation month, end to end

The following deliberately short pseudocode connects the earlier sections. `fit_transform(train)` creates leakage-safe training features; `transform(homes)` looks up history for the future homes. The resulting tables are what a tree model actually consumes.


In [ ]:
def one_month_in_plain_pseudocode(train, homes, feature_builder, history_encoder, model, target):
    base_train = feature_builder.fit_transform(train)
    base_homes = feature_builder.transform(homes)
    X_train = pd.concat([base_train, history_encoder.fit_transform(train)], axis=1)
    X_homes = pd.concat([base_homes, history_encoder.transform(homes)], axis=1)
    model.fit(X_train, target)
    return model.predict(X_homes)


The real workflow runs the required component versions of that pattern, converts their log outputs to dollars, and blends them. Then it advances the cutoff by one month and rebuilds everything from the newly allowed history. See [`experiments/ensemble_walk_forward.py`](../experiments/ensemble_walk_forward.py), [`experiments/xgboost_walk_forward.py`](../experiments/xgboost_walk_forward.py), and [`finalize_results.py`](../finalize_results.py).


## 11. Final training and saved model

At the final June 1, 2026 cutoff, the off-market tree models use all labeled sales from January 2021 through May 2026. The off-market text model and every list-aware model use the trailing 36 months, approximately June 2023 through May 2026. All encoders, category mappings, vectorizers, fitted models, blend weights, and calibration factors are saved together in one checkpoint.


In [ ]:
final_cutoff = pd.Timestamp('2026-06-01')
all_allowed_history = listings[listings.date_sold < final_cutoff]
last_36_months = all_allowed_history[
    all_allowed_history.date_sold >= final_cutoff - pd.DateOffset(months=36)
]


The saved checkpoint predicts without retraining. This is important: feature transformations for new homes must use the exact category lists, historical summaries, word vocabulary, and model parameters learned at final training time.

Operational entry points:

- Train once and save: [`train_final.py`](../train_final.py)
- Load and predict without retraining: [`predict_artifact.py`](../predict_artifact.py)
- Recompute walk-forward validation: [`validate.py`](../validate.py)
- Short run instructions and selected checkpoint: [`FINAL_MODEL.md`](../FINAL_MODEL.md)
- Full experiment history and rejected ideas: [`EXPERIMENT_REPORT.md`](EXPERIMENT_REPORT.md)


## Mental model to keep

For each home, the system asks three kinds of questions: **What is this property like? What did similar nearby homes sell for before the cutoff? What extra signal is present in the remarks or list price?** The engineered columns make those answers available to the models. The tree and text models learn different ways of turning them into prices, and the weighted blend reduces reliance on any one model's mistakes.
